# 12.1 - Architecture Patterns for AI Apps

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook works through Architecture Patterns hands-on: The API gateway: one front door; The model router: the right model per request; Provider fallback: never go down; Queue-based processing: decouple intake from work; Streaming: time-to-first-token; The tier decision: monolith, microservices, or serverless.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Every pattern in this notebook is pure engineering logic, so almost nothing here needs a network call or a key. This first cell just makes sure `python-dotenv` is available for loading environment variables - uncomment the line on Colab or a fresh machine.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

A dependency-install cell, not a model call. Only `python-dotenv` is needed, and it is commented out so a normal local run does nothing.

**How the code works - step by step**
- **`pip install python-dotenv`** - the one third-party package, used only to read a `.env` file if you have one; left commented so it runs only when you uncomment it.

**In one line:** optional environment prep - almost every cell below runs with no key.

**What you'll see:** (no output - environment prep)

## Setup - API keys

The multi-provider demos read from the environment rather than hardcoding secrets. This cell registers empty placeholders for the three provider keys so later code can reference them without a `KeyError`; any one real key is enough, and most cells need none at all.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A configuration cell that seeds three environment variables to empty strings via `setdefault`, so nothing is overwritten if a key is already set. No provider is actually called here.

**How the code works - step by step**
- **`import os`** - gives access to `os.environ`, the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** - registers the OpenAI key slot only if it is not already present.
- **`ANTHROPIC_API_KEY` / `GOOGLE_API_KEY`** - the same for Anthropic and Google, each with a comment pointing at where to get the key.

**In one line:** declare the three key slots from the environment, never from source code.

**What you'll see:** (no output - environment prep)

## 1 - The API gateway: one front door

Every production request should pass through a single layer that authenticates the caller, enforces a rate limit, logs the call, and routes it to the right service - instead of re-implementing those four concerns on every endpoint. This cell builds that layer as one function and drives a batch of requests through it, using a tick counter in place of a real clock so the result is deterministic and keyless.

In [ ]:
# THE API GATEWAY - one front door for every AI feature. Auth, rate limiting, logging, and
# routing all live in ONE layer instead of being re-implemented on every endpoint. We drive
# a batch of requests through it deterministically (a tick clock stands in for real time). No key.

API_KEYS = {"key-acme": "acme-corp"}      # valid keys -> tenant
RATE_LIMIT, WINDOW = 3, 10                 # 3 requests per 10 ticks, per key
seen = {}                                   # key -> list of request ticks (sliding window)
ROUTES = {"/v1/chat": "chat-service", "/v1/embed": "embedding-service", "/v1/image": "imagen-service"}

def gateway(tick, api_key, path):
    if api_key not in API_KEYS:
        return 401, "invalid API key"                      # AUTH
    hits = [t for t in seen.get(api_key, []) if tick - t < WINDOW]  # RATE LIMIT (sliding window)
    if len(hits) >= RATE_LIMIT:
        return 429, "rate limit exceeded"
    hits.append(tick); seen[api_key] = hits
    if path not in ROUTES:
        return 404, "no route"
    return 200, f"routed -> {ROUTES[path]}"                 # ROUTE

requests = [
    (0, "key-acme", "/v1/chat"),
    (1, "key-acme", "/v1/embed"),
    (2, "key-acme", "/v1/chat"),
    (3, "key-acme", "/v1/chat"),    # 4th within the window -> 429
    (4, "key-bad",  "/v1/chat"),    # bad key -> 401
]
print("Every request passes through the gateway (auth -> rate limit -> route), and is LOGGED:")
for tick, key, path in requests:
    status, msg = gateway(tick, key, path)
    print(f"  [t={tick}] {key:9} {path:9} -> {status} {msg}")
print("\nOne layer handles auth, rate limiting, logging, and routing for ALL AI features.")
# Output:
# Every request passes through the gateway (auth -> rate limit -> route), and is LOGGED:
#   [t=0] key-acme  /v1/chat  -> 200 routed -> chat-service
#   [t=1] key-acme  /v1/embed -> 200 routed -> embedding-service
#   [t=2] key-acme  /v1/chat  -> 200 routed -> chat-service
#   [t=3] key-acme  /v1/chat  -> 429 rate limit exceeded
#   [t=4] key-bad   /v1/chat  -> 401 invalid API key
#
# One layer handles auth, rate limiting, logging, and routing for ALL AI features.

**Explanation**

One `gateway()` function that fixes the order of the four cross-cutting concerns: auth, then a sliding-window rate limit, then logging, then routing. The driver loop feeds it five hand-picked requests designed to trip each outcome (200, 429, 401).

**How the code works - step by step**
- **`API_KEYS`, `ROUTES`** - lookup tables mapping valid keys to a tenant and paths to a backend service.
- **`gateway(tick, api_key, path)`** - runs the checks in order: rejects an unknown key with **401**, counts hits inside the `WINDOW` and rejects the 4th with **429**, rejects an unknown path with **404**, otherwise returns **200** with the routed service.
- **`seen`** - a per-key list of request ticks; the sliding window keeps only hits newer than `WINDOW` ticks.
- **the request loop** - sends 3 valid `key-acme` calls, a 4th inside the window, and one `key-bad` call, printing each status as the log line.

**In one line:** auth -> rate-limit -> log -> route, once, for every AI feature.

**What you'll see:** Five log lines: the first three `key-acme` requests return 200 and route to chat/embedding services, the 4th returns `429 rate limit exceeded`, and `key-bad` returns `401 invalid API key`.

## 2 - The model router: the right model per request

Most production traffic is simple - greetings, lookups, short answers - so sending all of it to an expensive frontier model is pure waste. This cell classifies each query's complexity with a cheap heuristic, routes simple queries to a fast/cheap tier and hard ones to a frontier tier, and computes the cost saving versus sending everything to the frontier.

In [ ]:
# THE MODEL ROUTER - the right model per request. Most traffic is simple; sending every
# query to a frontier model is wasteful. Classify complexity, route simple -> a fast/cheap tier
# and hard -> a frontier tier. A cheap heuristic classifier stands in for a real one. No key.
# Prices are ILLUSTRATIVE per-query blended costs.

FAST     = {"tier": "fast/cheap", "cost": 0.001}     # e.g. a Haiku/Flash/Nano-class model
FRONTIER = {"tier": "frontier",   "cost": 0.020}     # e.g. a Sonnet/Opus/Pro-class model
HARD_SIGNALS = ("compare", "analyze", "why", "design", "code", "step by step", "npv", "trade-off")

def classify(q):
    ql = q.lower()
    if len(q) > 60 or any(s in ql for s in HARD_SIGNALS):
        return "complex"
    return "simple"

def route(q):
    return FRONTIER if classify(q) == "complex" else FAST

queries = ["Hi there!", "What are your hours?",
           "Compare the ROI of this course vs a CS masters with a 5-year NPV"]
print("Route each query by classified complexity:")
router_cost = 0.0
for q in queries:
    m = route(q)
    router_cost += m["cost"]
    print(f"  [{classify(q):7}] {q[:44]:44} -> {m['tier']} (${m['cost']}/query)")

all_frontier = len(queries) * FRONTIER["cost"]
saving = (all_frontier - router_cost) / all_frontier
print(f"\nrouter cost ${router_cost:.3f} vs all-frontier ${all_frontier:.3f} -> {saving:.0%} cheaper on THIS mix.")
print("At production ratios (roughly 90% of traffic simple), tiered routing saves 80-90%.")
# Output:
# Route each query by classified complexity:
#   [simple ] Hi there!                                    -> fast/cheap ($0.001/query)
#   [simple ] What are your hours?                         -> fast/cheap ($0.001/query)
#   [complex] Compare the ROI of this course vs a CS maste -> frontier ($0.02/query)
#
# router cost $0.022 vs all-frontier $0.060 -> 63% cheaper on THIS mix.
# At production ratios (roughly 90% of traffic simple), tiered routing saves 80-90%.

**Explanation**

A routing-plus-cost-measurement harness. A heuristic `classify()` stands in for a real classifier, and the prices are illustrative per-query blended costs used only to show the saving arithmetic.

**How the code works - step by step**
- **`FAST` / `FRONTIER`** - the two tiers, with illustrative per-query costs ($0.001 vs $0.020).
- **`classify(q)`** - marks a query `complex` if it is over 60 characters or contains a hard signal (`compare`, `analyze`, `why`, `npv`, `code`, ...); otherwise `simple`.
- **`route(q)`** - returns the frontier tier for complex queries, the fast tier otherwise.
- **the loop + cost meter** - routes three queries (two greetings, one ROI comparison), sums `router_cost`, and compares it against `all_frontier` to compute the percentage saved.

**In one line:** classify complexity -> route simple to cheap, hard to frontier -> measure the saving.

**What you'll see:** The two greetings print as `simple -> fast/cheap` and the ROI comparison as `complex -> frontier`, then: router cost $0.022 vs all-frontier $0.060, i.e. 63% cheaper on this mix (80-90% at real production ratios).

## 3 - Provider fallback: never go down

Every LLM provider has outages, so an app that calls only one shares that provider's bad day. This cell lists providers in priority order and tries them in turn - the first success wins - so when the primary is down the request quietly fails over to the next and the user never sees an error.

In [ ]:
# PROVIDER FALLBACK - never go down. Try providers in priority order; the first success wins.
# Because any single provider has outages, a multi-provider chain gives zero user-visible
# downtime. We SCRIPT which providers are up (a real call would try the network). No key.

class Fallback:
    def __init__(self, providers):
        self.providers = sorted(providers, key=lambda p: p["priority"])
        self.stats = {p["name"]: {"ok": 0, "fail": 0} for p in providers}

    def call(self, prompt):
        for p in self.providers:
            if p["up"]:
                self.stats[p["name"]]["ok"] += 1
                return {"text": f"{p['name']}: {prompt[:24]}", "via": p["name"]}
            self.stats[p["name"]]["fail"] += 1
            print(f"  [{p['name']}] DOWN -> trying next...")
        return {"text": "all providers failed", "via": None}

# gemini is down today; openai and claude are up.
chain = Fallback([
    {"name": "gemini", "up": False, "priority": 1},
    {"name": "openai", "up": True,  "priority": 2},
    {"name": "claude", "up": True,  "priority": 3},
])
print("Call with the primary (gemini) down:")
r = chain.call("What is the refund policy?")
print(f"  -> served by {r['via']}: {r['text']}")
print(f"  stats: {chain.stats}")
print("The user never saw an error. (A circuit breaker to stop hammering gemini is Lesson 12.2.)")
# Output:
# Call with the primary (gemini) down:
#   [gemini] DOWN -> trying next...
#   -> served by openai: openai: What is the refund polic
#   stats: {'gemini': {'ok': 0, 'fail': 1}, 'openai': {'ok': 1, 'fail': 0}, 'claude': {'ok': 0, 'fail': 0}}
# The user never saw an error. (A circuit breaker to stop hammering gemini is Lesson 12.2.)

**Explanation**

A small `Fallback` class that scripts which providers are up (a real version would try the network) and tracks per-provider success/failure stats. The demo forces the primary down to show the failover.

**How the code works - step by step**
- **`__init__`** - sorts the providers by `priority` and creates an `ok`/`fail` counter for each.
- **`call(prompt)`** - walks the sorted chain, returns the first provider whose `up` is true (incrementing its `ok`), and logs+counts a failure for each one it skips.
- **the chain** - gemini (priority 1) is down; openai and claude are up.
- **the all-failed path** - if no provider is up, it returns one clear `all providers failed` result instead of crashing.

**In one line:** try providers in priority order, take the first success, count the outcome.

**What you'll see:** It logs `[gemini] DOWN -> trying next...`, then `served by openai`, and prints the stats dict recording gemini's one failure and openai's one success - the user saw no error.

## 4 - Queue-based processing: decouple intake from work

Some AI work is slow, and holding an HTTP connection open for minutes is fragile - it times out and ties up a server. This cell splits intake from work: `submit` accepts the request and returns a `job_id` instantly, a background worker advances the job, and the client polls a status endpoint until it is completed. A tick clock stands in for real time.

In [ ]:
# QUEUE-BASED PROCESSING - decouple intake from work. Accept the request INSTANTLY (return a
# job_id in well under a second), enqueue it, let independently-scaled workers process it, and
# let the client POLL for status. This absorbs traffic spikes. A tick clock stands in for time. No key.

jobs = {}                        # job_id -> {status, result}
def submit(prompt):              # the API: returns immediately
    job_id = f"job-{len(jobs)+1}"
    jobs[job_id] = {"status": "queued", "prompt": prompt}
    return job_id
def worker_tick(job_id, tick):   # a background worker advances the job
    j = jobs[job_id]
    if tick == 1: j["status"] = "processing"
    if tick == 4: j.update(status="completed", result=f"generated: {j['prompt'][:20]}")
def poll(job_id):                # the client status endpoint
    return jobs[job_id]["status"]

jid = submit("write a 500-word product description")   # returns at t=0, ~1 tick
print(f"submit -> {jid} returned at t=0 (the API never blocks on the work)")
for t in range(5):
    worker_tick(jid, t)
    if t in (0, 1, 4):
        print(f"  poll at t={t}: {poll(jid)}")
print("\nIntake took ~1 tick; the work took 4. They scale independently. (This is 11.9 at the arch layer.)")
# Output:
# submit -> job-1 returned at t=0 (the API never blocks on the work)
#   poll at t=0: queued
#   poll at t=1: processing
#   poll at t=4: completed
#
# Intake took ~1 tick; the work took 4. They scale independently. (This is 11.9 at the arch layer.)

**Explanation**

A submit-poll lifecycle simulation, not a real queue. Three tiny functions play the three roles - the API, the worker, and the client status endpoint - against a shared `jobs` dict.

**How the code works - step by step**
- **`submit(prompt)`** - the API: creates a `queued` job and returns its `job_id` at t=0, never blocking on the work.
- **`worker_tick(job_id, tick)`** - a background worker that flips the job to `processing` at t=1 and `completed` at t=4.
- **`poll(job_id)`** - the `GET /status/{id}` endpoint the client calls to read the current status.
- **the loop** - advances the worker across ticks 0-4 and polls at t=0, 1, 4 to show the state transitions.

**In one line:** submit returns instantly -> workers process on their own clock -> client polls to completion.

**What you'll see:** `submit -> job-1 returned at t=0`, then polls reading `queued` (t=0), `processing` (t=1), `completed` (t=4) - intake took ~1 tick, the work took 4, and they scale independently.

## 5 - Streaming: time-to-first-token

An LLM produces tokens one at a time over several seconds; buffering the whole reply makes a fast model feel broken behind a spinner. This cell contrasts SSE streaming (emit each token as it is produced) with batch delivery (return everything at once) and measures when the user first sees anything - the metric that actually matters is time-to-first-token.

In [ ]:
# STREAMING - time-to-first-token, not total latency, drives the experience. An LLM generates
# tokens sequentially over seconds; buffering the whole reply feels slow. SSE streams each token
# as it is produced, so the first word lands almost immediately. A tick clock = time. No key.

reply = "Here is the answer to your question about refunds".split()
N = len(reply)

# STREAMING (SSE): emit token i at tick i+1; the user sees the first word at t=1.
print("STREAMING (SSE) - the user sees words appear:")
first_visible_stream = None
for i, tok in enumerate(reply):
    t = i + 1
    if first_visible_stream is None:
        first_visible_stream = t
print(f"  time-to-first-token = t={first_visible_stream}; full reply done at t={N}")

# BATCH: nothing is visible until the whole reply is buffered at t=N.
first_visible_batch = N
print("BATCH (buffer then return) - the user stares at a spinner:")
print(f"  first-visible = t={first_visible_batch} (the WHOLE reply at once)")

print(f"\nSSE shows the first word {first_visible_batch - first_visible_stream} ticks sooner "
      f"(t={first_visible_stream} vs t={first_visible_batch}). Same total time, far better perceived speed.")
print("SSE is the default for token streams (HTTP, CDN-friendly, auto-reconnect); WebSocket only for two-way.")
# Output:
# STREAMING (SSE) - the user sees words appear:
#   time-to-first-token = t=1; full reply done at t=9
# BATCH (buffer then return) - the user stares at a spinner:
#   first-visible = t=9 (the WHOLE reply at once)
#
# SSE shows the first word 8 ticks sooner (t=1 vs t=9). Same total time, far better perceived speed.
# SSE is the default for token streams (HTTP, CDN-friendly, auto-reconnect); WebSocket only for two-way.

**Explanation**

A perceived-latency comparison harness. It does not call a model - it walks a fixed 9-token reply and records the first-visible tick under each delivery strategy.

**How the code works - step by step**
- **`reply` / `N`** - a 9-word reply used as the token stream.
- **the streaming loop** - emits token `i` at tick `i+1` and records `first_visible_stream` at the very first token (t=1).
- **the batch case** - sets `first_visible_batch = N`, since nothing is visible until the whole reply is buffered at t=9.
- **the comparison** - prints how many ticks sooner SSE shows the first word, and notes SSE is the HTTP/CDN-friendly, auto-reconnecting default; WebSocket is only for two-way mid-stream traffic.

**In one line:** stream shows word one at t=1, batch shows everything at t=9 - same total time, very different feel.

**What you'll see:** Streaming reports time-to-first-token = t=1 (done at t=9); batch reports first-visible = t=9; the summary: SSE shows the first word 8 ticks sooner at the same total time.

## 6 - The tier decision: monolith, microservices, or serverless

Where do all these patterns live? The discipline is not over-engineering: default to a modular monolith because in-process calls are nanoseconds while inter-service calls cost milliseconds plus ops overhead, and decompose only on a real trigger - usually GPU inference that must scale on its own. This cell encodes those rules and routes four app profiles to a tier.

In [ ]:
# THE TIER DECISION - modular monolith vs microservices vs serverless. Start with a modular
# monolith (in-process calls are nanoseconds); decompose to microservices only when something -
# usually GPU inference - must scale on its own; serverless fits event-driven, bursty work. No key.

def choose(self_hosted_gpu, independent_scaling, event_driven, big_team):
    if self_hosted_gpu and independent_scaling:
        return "MICROSERVICES", "GPU inference must scale on its own hardware"
    if big_team:
        return "MICROSERVICES", "many teams need independent deploys"
    if event_driven and not self_hosted_gpu:
        return "SERVERLESS", "event-driven + bursty -> scale to zero (Cloud Run / RunPod)"
    return "MODULAR MONOLITH", "start here: one deploy, in-process calls"

profiles = [
    ("API-calling MVP, 4 engineers",            dict(self_hosted_gpu=False, independent_scaling=False, event_driven=False, big_team=False)),
    ("self-hosted 70B model, own scaling",      dict(self_hosted_gpu=True,  independent_scaling=True,  event_driven=False, big_team=False)),
    ("nightly webhook + batch pipeline",        dict(self_hosted_gpu=False, independent_scaling=False, event_driven=True,  big_team=False)),
    ("30-engineer multi-team platform",         dict(self_hosted_gpu=False, independent_scaling=False, event_driven=False, big_team=True)),
]
print("Route each app profile to a deployment tier:")
for name, props in profiles:
    tier, why = choose(**props)
    print(f"  {name:38} -> {tier:16} ({why})")
print("\nDefault to a modular monolith; decompose only when you must. Lambda has NO GPU (orchestration only).")
# Output:
# Route each app profile to a deployment tier:
#   API-calling MVP, 4 engineers           -> MODULAR MONOLITH (start here: one deploy, in-process calls)
#   self-hosted 70B model, own scaling     -> MICROSERVICES    (GPU inference must scale on its own hardware)
#   nightly webhook + batch pipeline       -> SERVERLESS       (event-driven + bursty -> scale to zero (Cloud Run / RunPod))
#   30-engineer multi-team platform        -> MICROSERVICES    (many teams need independent deploys)
#
# Default to a modular monolith; decompose only when you must. Lambda has NO GPU (orchestration only).

**Explanation**

A decision-function demo. `choose()` is a short priority ladder of if-statements mapping four boolean properties of an app to one of three deployment tiers, with the reason.

**How the code works - step by step**
- **`choose(self_hosted_gpu, independent_scaling, event_driven, big_team)`** - returns MICROSERVICES when GPU inference needs its own scaling or many teams need independent deploys, SERVERLESS for event-driven bursty work without self-hosted GPU, and MODULAR MONOLITH as the default.
- **`profiles`** - four app descriptions: an API-calling MVP, a self-hosted 70B model, a nightly webhook/batch pipeline, and a 30-engineer multi-team platform.
- **the loop** - runs each profile through `choose()` and prints the tier plus the one-line rationale.

**In one line:** start on a modular monolith; decompose only when a concrete trigger forces it.

**What you'll see:** Four routed profiles: the MVP -> MODULAR MONOLITH, the self-hosted 70B -> MICROSERVICES, the nightly pipeline -> SERVERLESS, the 30-engineer platform -> MICROSERVICES, plus the reminder that Lambda has no GPU.

## 7 - The reference architecture: all five, one request

Now assemble everything. A production request does not hit the model directly - it flows through gateway -> router -> cache -> provider(with fallback) -> stream, each layer owning exactly one concern. This cell traces a single chat request through every layer with illustrative per-layer latencies and shows where the time actually goes.

In [ ]:
# THE REFERENCE ARCHITECTURE - all five patterns in ONE request. Trace a single chat request
# through every layer and watch where the time goes. Latencies are ILLUSTRATIVE milliseconds.
# This is the whole system: gateway -> router -> cache -> provider(fallback) -> stream. No key.

def trace_request(prompt):
    t = 0; log = []
    def step(layer, ms, note):
        nonlocal t; t += ms; log.append((layer, ms, t, note))
    step("gateway",  2, "auth ok, rate ok, logged")
    step("router",   1, "classified complex -> frontier tier")
    step("cache",    1, "miss (a hit would return here; see 12.4)")
    step("provider", 300, "gemini DOWN -> fell back to openai (see 12.2)")
    step("stream",   40, "first SSE token sent to the client (TTFT)")
    return log

print("One chat request, traced through the reference architecture:")
for layer, ms, cum, note in trace_request("what is the refund policy?"):
    print(f"  {layer:9} +{ms:>3}ms  (t={cum:>3}ms)  {note}")
print("\nTime-to-first-token = t=344ms. The gateway, router, fallback, and stream each own one concern;")
print("together they turn model.generate() into a production system. Each layer deepens in its own M12 lesson.")
# Output:
# One chat request, traced through the reference architecture:
#   gateway   +  2ms  (t=  2ms)  auth ok, rate ok, logged
#   router    +  1ms  (t=  3ms)  classified complex -> frontier tier
#   cache     +  1ms  (t=  4ms)  miss (a hit would return here; see 12.4)
#   provider  +300ms  (t=304ms)  gemini DOWN -> fell back to openai (see 12.2)
#   stream    + 40ms  (t=344ms)  first SSE token sent to the client (TTFT)
#
# Time-to-first-token = t=344ms. The gateway, router, fallback, and stream each own one concern;
# together they turn model.generate() into a production system. Each layer deepens in its own M12 lesson.

**Explanation**

A tracing harness that accumulates a running total across the five layers. The latencies are illustrative milliseconds; the point is the shape - the network provider call dwarfs everything else.

**How the code works - step by step**
- **`trace_request(prompt)`** - defines an inner `step(layer, ms, note)` that adds each layer's latency to a running clock `t` and appends a log row.
- **the five steps** - gateway (+2ms, auth/rate/log), router (+1ms, classify to frontier), cache (+1ms, miss), provider (+300ms, gemini down -> fell back to openai), stream (+40ms, first SSE token).
- **the print loop** - shows each layer's added latency and the cumulative total, ending at the time-to-first-token.

**In one line:** gateway/router/cache are near-free; the provider round-trip dominates; TTFT is the number the user feels.

**What you'll see:** A five-line trace with cumulative times - gateway 2ms, router 3ms, cache 4ms, provider 304ms, stream 344ms - so time-to-first-token = 344ms, dominated by the provider network call.

## 8 - The standard stack: FastAPI + SSE

The real version of the gateway and stream is a FastAPI service returning a Server-Sent Events response. This cell is an illustrative minimal example of that stack - the production shape of Step 5's streaming and Step 1's endpoint.

> **Illustrative only** - it is marked `%%javascript` and won't execute as-is; running the real thing needs `pip install fastapi uvicorn` plus an ASGI server and a live `llm_client`.

In [ ]:
%%javascript
# THE STANDARD AI API STACK - FastAPI + SSE (illustrative minimal example).
# FastAPI (async, Pydantic validation, OpenAPI docs) is the default Python framework for LLM
# services; a StreamingResponse with media_type "text/event-stream" is the standard token stream.
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import json

app = FastAPI(title="netsetos-ai-gateway")

async def token_stream(prompt: str):
    async for token in llm_client.stream(prompt):        # iterate the LLM's streaming response
        yield f"data: {json.dumps({'token': token})}\n\n"  # one SSE frame per token
    yield "data: [DONE]\n\n"

@app.get("/v1/stream")
async def stream(prompt: str = "Hello"):
    return StreamingResponse(token_stream(prompt), media_type="text/event-stream")

# Browser client:  const es = new EventSource("/v1/stream?prompt=hi"); es.onmessage = e => ...
# For a MULTI-PROVIDER gateway (fallback + routing across OpenAI / Anthropic / Google behind one
# OpenAI-compatible endpoint) you put LiteLLM or Portkey in front - that is Lesson 12.3.
# Output: (illustrative minimal example - needs `pip install fastapi uvicorn` + an ASGI server.)
# GET /v1/stream returns an SSE stream of {"token": ...} frames ending in [DONE]; EventSource reads it.

**Explanation**

A reference code sketch, not a runnable cell. It shows the canonical FastAPI + SSE pattern: an async generator yielding one SSE frame per token, exposed on a streaming endpoint.

**How the code works - step by step**
- **`app = FastAPI(...)`** - the async framework (Pydantic validation, OpenAPI docs) that is the default for Python LLM services.
- **`token_stream(prompt)`** - an async generator that iterates the LLM's streaming response and yields each token as a `data: {...}\n\n` SSE frame, ending with `data: [DONE]\n\n`.
- **`@app.get("/v1/stream")`** - returns a `StreamingResponse` with `media_type="text/event-stream"`, the standard SSE content type.
- **the client comment** - a browser `EventSource` reads the frames; a multi-provider gateway (LiteLLM/Portkey) sits in front, which is Lesson 12.3.

**In one line:** FastAPI + a `StreamingResponse` of `text/event-stream` frames is the production form of the gateway and stream.

**What you'll see:** No live output - it is an illustrative sketch. Conceptually, `GET /v1/stream` returns an SSE stream of `{"token": ...}` frames ending in `[DONE]`, which a browser `EventSource` consumes.

Seven small, keyless programs turned one fragile `model.generate()` call into a production system: a gateway that authenticates, rate-limits, logs, and routes; a router that sends cheap traffic to a cheap model; a fallback chain that survives an outage; a queue that absorbs slow work; SSE streaming that makes the first token land fast; and a tier decision that keeps you on a modular monolith until GPU inference forces a split. The final trace shows all of them in one request, with the provider network call dominating the latency and time-to-first-token as the number the user actually feels. Every layer here gets its own deep lesson across the rest of Module 12 - reliability (12.2), gateways and routing (12.3), caching (12.4), containers (12.5), CI/CD (12.6), scaling and serverless-GPU (12.7), monitoring (12.8).